# Synthcity Evaluation — Google Colab

Runs `eval_synthcity.py` one method at a time to avoid OOM.

**Before starting:** Switch runtime to T4 GPU (Runtime → Change runtime type → T4 GPU).

**Steps:**
1. Run Cell 1 (install) — runtime will restart automatically
2. After restart, run Cell 2 (mount Drive + paths)
3. Run Cell 3 (upload eval script)
4. Run Cells 4–7 one at a time — each method clears memory before the next

In [ ]:
# Cell 1 — Install (runtime will restart after this)
!pip install synthcity -q
!pip install 'opacus<1.5' -q
!pip install 'pandas>=2.1,<3.0' -q

import os
os.kill(os.getpid(), 9)  # force restart to clear numpy binary conflict

In [ ]:
# Cell 2 — Mount Drive + set up working directory
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/VRI/experimentation'

os.chdir('/content')

# Symlink data folders from Drive into /content so the script finds them
for folder in ['data', 'synthetic_data']:
    if not os.path.exists(f'/content/{folder}'):
        os.symlink(f'{DRIVE_BASE}/{folder}', f'/content/{folder}')

os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/evaluation', exist_ok=True)

print('Drive mounted. Folders:')
print('  data:          ', os.listdir('/content/data')[:5])
print('  synthetic_data:', os.listdir('/content/synthetic_data'))

## Cell 3 — Upload eval script

In [ ]:
# Cell 3b — Copy eval script from Drive
import shutil
shutil.copy(
    f'{DRIVE_BASE}/evaluation/eval_synthcity.py',
    '/content/evaluation/eval_synthcity.py'
)
print('Copied eval_synthcity.py')

In [ ]:
# Bayesian Network
!python evaluation/eval_synthcity.py bayesian_network

In [ ]:
# PrivBayes
!python evaluation/eval_synthcity.py privbayes

In [ ]:
# CTGAN
!python evaluation/eval_synthcity.py ctgan

In [ ]:
# DPGAN
!python evaluation/eval_synthcity.py dpgan

In [ ]:
# Cell 8 — Save results back to Drive
import shutil

shutil.copy('/content/results/synthcity_results.csv',
            f'{DRIVE_BASE}/results/synthcity_results.csv')
shutil.copy('/content/evaluation/eval_synthcity_log.txt',
            f'{DRIVE_BASE}/evaluation/eval_synthcity_log.txt')
shutil.copy('/content/evaluation/eval_overhead.csv',
            f'{DRIVE_BASE}/evaluation/eval_overhead.csv')

print('Results saved to Drive.')

import pandas as pd
print(pd.read_csv('/content/results/synthcity_results.csv', index_col=0))